In [ ]:
!pip install transformers

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import transformers
from transformers import  AutoTokenizer, AutoModel, BertTokenizerFast, BertModel, XLMRobertaModel, RobertaTokenizer, DistilBertModel, DistilBertTokenizer, BertConfig, BertForSequenceClassification
#from normalizer import normalize
# specify GPU
import datetime


In [ ]:
df = pd.read_json("/kaggle/input/datasets/swagotodas/newsclassification2/ittefaq.jsonl",lines=True)
df.head()

In [ ]:
df.shape

In [ ]:
df['Category'].value_counts()

In [ ]:
df['Content'].isna().sum()

In [ ]:
df = df.dropna(subset=['Category'])

In [ ]:
df = df.dropna(subset=['Content'])

In [ ]:
df['Category'].value_counts()

In [ ]:
counts = df['Category'].value_counts()
df = df[df['Category'].isin(counts[counts > 1].index)]  # remove classes with 1 sample 

#REmoving class PSI as it has jsut 1 sample

train_text, temp_text, train_labels, temp_labels = train_test_split(df['Content'], df['Category'],
                                                                    random_state=2018,
                                                                    test_size=0.2,
                                                                    stratify=df['Category']
                                                                    )

# i will use temp_text and temp_labels to create validation and test set
val_text, test_text, val_labels, test_labels = train_test_split(temp_text, temp_labels,
                                                                random_state=2018,
                                                                test_size=0.2,
                                                                stratify=temp_labels
                                                                )

In [ ]:
print(set(train_labels))
print(set(val_labels))
print(set(test_labels))

In [ ]:
from transformers import AutoTokenizer, AutoModel

model_name = "csebuetnlp/banglabert"

tokenizer = AutoTokenizer.from_pretrained(model_name)
bert = AutoModel.from_pretrained(model_name)


In [ ]:
#Encoding the label

from sklearn.preprocessing import LabelEncoder
import torch

le = LabelEncoder()
train_labels_enc = le.fit_transform(train_labels)
val_labels_enc   = le.transform(val_labels)
test_labels_enc  = le.transform(test_labels)

num_labels = len(le.classes_)

print(sorted(set(train_labels_enc)))

In [ ]:
class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts.tolist()
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(self.labels[idx],dtype=torch.long)
        }

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
max_seq_len = 128
train_dataset = NewsDataset(train_text, train_labels_enc, tokenizer, max_seq_len)
val_dataset   = NewsDataset(val_text, val_labels_enc, tokenizer, max_seq_len)
test_dataset  = NewsDataset(test_text, test_labels_enc, tokenizer, max_seq_len)

In [ ]:
from torch.utils.data import DataLoader


batch_size = 16

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader   = DataLoader(val_dataset, batch_size=batch_size)
test_dataloader  = DataLoader(test_dataset, batch_size=batch_size)

for batch in train_dataloader:
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['label'].to(device)

In [ ]:

class BERT_Arch(nn.Module):

    def __init__(self, bert):

      super(BERT_Arch, self).__init__()

      self.bert = bert

      # dropout layer
      self.dropout = nn.Dropout(0.1)

      # relu activation function
      self.relu =  nn.ReLU()

      # dense layer 1
      self.fc1 = nn.Linear(768,512)

      # dense layer 2 (Output layer)
      self.fc2 = nn.Linear(512,num_labels)             #there is 16 classes

      #softmax activation function
      self.softmax = nn.LogSoftmax(dim=1)

    #define the forward pass
    def forward(self, sent_id, mask):

      #pass the inputs to the model
     # _, cls_hs = self.bert(sent_id, attention_mask=mask)
      outputs = self.bert(sent_id,attention_mask=mask)
      cls_hs = outputs.last_hidden_state[:, 0, :]   # use CLS token  
      x = self.fc1(cls_hs)

      x = self.relu(x)

      x = self.dropout(x)

      # output layer
      x = self.fc2(x)

      # apply softmax activation
      x = self.softmax(x)

      return x

In [ ]:
print(len(set(train_labels)))

In [ ]:
model = BERT_Arch(bert)
print(model)

# Freeze bottom 6 layers
for i in range(0, 6):
    for p in model.bert.encoder.layer[i].parameters():
        p.requires_grad = False

# Unfreeze top 6 layers
for i in range(6, 12):
    for p in model.bert.encoder.layer[i].parameters():
        p.requires_grad = True

#Find out the parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print("Trainable parameters:", trainable)
print("Total parameters:", total)


# push the model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Wrap the model with DataParallel if multiple GPUs are available
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs for training.")
    model = nn.DataParallel(model)
    


<!-- Adding new block below instead of the upper one for grdual unfreezing -->

In [ ]:
# model = BERT_Arch(bert)
# print(model)

# for p in model.bert.encoder.parameters():
#     p.requires_grad = False
# for i in range(10, 12):
#     for p in model.bert.encoder.layer[i].parameters():
#         p.requires_grad = True

# #Find out the parameters
# trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
# total = sum(p.numel() for p in model.parameters())

# print("Trainable parameters:", trainable)
# print("Total parameters:", total)


# # push the model to GPU
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = model.to(device)

# # Wrap the model with DataParallel if multiple GPUs are available
# if torch.cuda.device_count() > 1:
#     print(f"Using {torch.cuda.device_count()} GPUs for training.")
#     model = nn.DataParallel(model)
    


In [ ]:
from torch.optim import AdamW
# define the optimizer
# optimizer = AdamW(model.parameters(), lr = 1e-5)

optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-5)

In [ ]:
from collections import Counter
# class_counts = Counter(train_labels)        #changed to Counter from np.bicount

from sklearn.utils import class_weight
#calculate class counts from encoded training labels
class_counts_encoded = Counter(train_labels_enc)
unique_classes = sorted(class_counts_encoded.keys())
# Calculate the uniform class weights
# num_classes = len(class_counts)
# class_weights_uniform = np.ones(num_classes) / num_classes

#compute class weights using sklearns compute_class weights
#balanced mode adjust weights

class_weights=class_weight.compute_class_weight(
    class_weight='balanced',
    classes= np.array(unique_classes),
    y = train_labels_enc
)

# Convert the class weights to a PyTorch tensor
# class_weights_uniform_tensor = torch.tensor(class_weights_uniform, dtype=torch.float32)
# class_weights_uniform_tensor = class_weights_uniform_tensor.to(device)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)
class_weights_tensor = class_weights_tensor.to(device)

In [ ]:
# loss function
cross_entropy  = nn.NLLLoss(weight=class_weights_tensor)

# number of training epochs
epochs = 10   #  i made it 4



In [ ]:
def train():
    model.train()
    total_loss = 0
    total_preds = []

    for step, batch in enumerate(train_dataloader):
        if step % 50 == 0 and not step == 0:
            print('  Batch {:>5,}  of  {:>5,}.'.format(step, len(train_dataloader)))

        # Correctly move tensors from the dictionary to the device
        sent_id = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        model.zero_grad()

        # Get model predictions
        preds = model(sent_id, mask)

        # Compute loss
        loss = cross_entropy(preds, labels)
        total_loss = total_loss + loss.item()

        loss.backward()

        # Clip gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        preds = preds.detach().cpu().numpy()
        total_preds.append(preds)

    avg_loss = total_loss / len(train_dataloader)
    total_preds = np.concatenate(total_preds, axis=0)

    return avg_loss, total_preds

In [ ]:
def evaluate():
    print("\nEvaluating...")
    model.eval()
    total_loss = 0
    total_preds = []

    for step, batch in enumerate(val_dataloader):
        if step % 50 == 0 and not step == 0:
            print('  Batch {:>5,}  of  {:>5,}.'.format(step, len(val_dataloader)))

        # Correctly move tensors from the dictionary to the device
        sent_id = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        with torch.no_grad():
            preds = model(sent_id, mask)
            loss = cross_entropy(preds, labels)
            total_loss = total_loss + loss.item()
            
            preds = preds.detach().cpu().numpy()
            total_preds.append(preds)

    avg_loss = total_loss / len(val_dataloader)
    total_preds = np.concatenate(total_preds, axis=0)

    return avg_loss, total_preds

In [ ]:
best_valid_loss = float('inf')

train_losses = []
valid_losses = []

patience = 1          # number of epochs to wait
counter = 0           # counts how many times val loss didn't improve

for epoch in range(epochs):

    print('\n Epoch {:} / {:}'.format(epoch + 1, epochs))

    train_loss, _ = train()
    valid_loss, _ = evaluate()

    # check improvement
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        counter = 0   # reset counter

        torch.save(model.state_dict(), 'saved_weights.pt')
        print("Model improved, saving...")
    else:
        counter += 1
        print(f"No improvement ({counter}/{patience})")

    train_losses.append(train_loss)
    valid_losses.append(valid_loss)

    print(f'\nTraining Loss: {train_loss}')
    print(f'Validation Loss: {valid_loss}')

    # EARLY STOPPING CONDITION
    if counter >= patience:
        print("Early stopping triggered")
        break

<!-- Adding next two blocks instead of previous one for gradual unfreezing technique -->

In [ ]:
# STAGE2_EPOCH = 3
# STAGE3_EPOCH = 6

In [ ]:
# def rebuild_optimizer(model):
#     return torch.optim.AdamW(
#         filter(lambda p: p.requires_grad, model.parameters()), lr=2e-5
#     )

# optimizer = rebuild_optimizer(model)

# best_valid_loss = float('inf')
# train_losses = []
# valid_losses = []
# patience = 2
# counter = 0

# for epoch in range(epochs):
#     print('\n Epoch {:} / {:}'.format(epoch + 1, epochs))

#     if epoch == STAGE2_EPOCH:
#         for i in range(6, 12):
#             for p in model.bert.encoder.layer[i].parameters():
#                 p.requires_grad = True
#         optimizer = rebuild_optimizer(model)
#         print("Stage 2: Unfrozen layers 6-11")

#     elif epoch == STAGE3_EPOCH:
#         for p in model.bert.encoder.parameters():
#             p.requires_grad = True
#         optimizer = rebuild_optimizer(model)
#         print("Stage 3: All layers unfrozen")

#     train_loss, _ = train()
#     valid_loss, _ = evaluate()

#     if valid_loss < best_valid_loss:
#         best_valid_loss = valid_loss
#         counter = 0
#         torch.save(model.state_dict(), 'saved_weights.pt')
#         print("Model improved, saving...")
#     else:
#         counter += 1
#         print(f"No improvement ({counter}/{patience})")

#     train_losses.append(train_loss)
#     valid_losses.append(valid_loss)
#     print(f'\nTraining Loss: {train_loss}')
#     print(f'Validation Loss: {valid_loss}')

#     if counter >= patience:
#         print("Early stopping triggered")
#         break

In [ ]:
path = 'saved_weights.pt'
model.load_state_dict(torch.load(path))

In [ ]:
# model.load_state_dict(torch.load("/kaggle/input/saved-weights-for-newsclass/saved_weights.pt", map_location=device))

In [ ]:
model.eval()
all_preds = []

with torch.no_grad():
    for batch in test_dataloader:
        batch_seq = batch['input_ids'].to(device)
        batch_mask = batch['attention_mask'].to(device)

        preds = model(batch_seq, batch_mask)

        preds = preds.detach().cpu().numpy()
        all_preds.append(preds)

all_preds = np.concatenate(all_preds, axis=0)

In [ ]:
# Convert logits/probabilities to predicted class indices
pred_labels = np.argmax(all_preds, axis=1)

# Build confusion matrix
pd.crosstab(test_labels_enc, pred_labels, rownames=['Actual'], colnames=['Predicted'])

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, f1_score

print("Accuracy:", accuracy_score(test_labels_enc, pred_labels))
print("Macro F1:", f1_score(test_labels_enc, pred_labels, average='macro'))
print("\nClassification Report:\n")
print(classification_report(test_labels_enc, pred_labels, digits=4))

In [ ]:
text = ["বিয়ন্সে! পশ্চাত্যের সঙ্গীতে সুপারস্টার। বছরের পর বছর বিলবোর্ডে দাপিয়ে বেড়ানো এই তারকা এবার নতুন ইতিহাস গড়লেন। গ্র্যামি অ্যাওয়ার্ডে সবচেয়ে বেশি পুরস্কার জেতার এক অনন্য রেকর্ড গড়েছেন  ৩৯ বছরের এই মার্কিন গায়িকা। পপ গানে দুনিয়া মাতালেও বিয়ন্সে বিচরণ করেছেন সঙ্গীত রচনা, অভিনয়, পরিচালনা থেকে শুরু করে মানবাধিকার রক্ষায়। বিয়ন্সের মতোই এবারের গ্র্যামিতে ইতিহাস গড়েছেন আরেক জনপ্রিয় গায়িকা টেইলর সুইফট। এই নিয়ে তিনবার জিতেছেন বর্ষসেরা অ্যালবামের পুরস্কার। বাংলাদেশ সময় সোমবার সকালে যুক্তরাষ্ট্রের লস অ্যাঞ্জেলেসের স্টেপলস সেন্টারে বসেছিল এবারের ৬৩তম গ্রামি আসর।সঙ্গীতের সর্বোচ্চ সম্মাননার এবারের অনুষ্ঠান মঞ্চ যেন সেজে উঠেছে গায়িকাদের অতুল কীর্তিকে সম্মান জানানোর জন্য। কেন না, এই বছরে সেরা পুরস্কারগুলোর সিংহভাগ দখল করেছেন গায়িকারাই! ব্ল্যাক প্যারেড অ্যালবামটি দিয়ে ২৮তম গ্র্যামি অ্যাওয়ার্ড জিতেন বিয়ন্সে। তিনিই এখন সবচেয়ে বেশি গ্র্যামি জয়ী তারকা। সবচেয়ে বেশি গ্রামি পুরস্কার জেতায় গায়িকা অ্যালিসন ক্রাউসের রেকর্ড এখন বিয়ন্সের দখলে।আরেক তারকা টেইলর সুইফটের লকডাউনের মধ্যেই তৈরি ফোকলোর অ্যালবাম তাঁকে এনে দিয়েছে অ্যালবাম অফ দ্য ইয়ারের পুরস্কার। তিনিই প্রথম গায়িকা হিসাবে এমন কোন কীর্তি দেখালেন। রেকর্ড অফ দ্য ইয়ার বিলি আইলিশের আর আই কান্ট ব্রেথ গানটি হলো সং অফ দ্য ইয়ার। বছরের সেরা নতুন শিল্পী হলেন মেগান থি স্টেলিয়ন। বেস্ট পপ সোলো পারফর্মেন্সের জন্য গ্র্যামি পান হ্যারি স্টাইল।মনোনয়নে থাকলেও কোন পুরস্কার ঘরে নিতে পারেনি কোরীয় পপ ব্যান্ড বিটিএস। তবে তাদের পারফর্মেন্সে একটুও হতাশ হয়নি দর্শকরা। স্টেপলস সেন্টারে হওয়া ৬৩তম গ্র্যামি আসরের মঞ্চ মাতান টেইলর সুইফট, লেডি গাগা, আরিয়ানা গ্রান্ডে, ডুয়া লিপা। আর উপস্থাপনায় ছিলেন ট্রেভর নোয়াহ।"]

In [ ]:
tokens= tokenizer(
    text,
    max_length = max_seq_len,
    padding='max_length',
    truncation=True,
    return_token_type_ids=False
)

In [ ]:
seq = torch.tensor(tokens['input_ids'])
mask = torch.tensor(tokens['attention_mask'])

In [ ]:
for i, class_name in enumerate(le.classes_):
    print(f"Class {i}: {class_name}")

In [ ]:
import torch.nn.functional as F

with torch.no_grad():
    preds = model(seq.to(device), mask.to(device))
    probs = F.softmax(preds, dim=1)   # convert logits to probabilities
    probs = probs.detach().cpu().numpy()

pred_labels = np.argmax(probs, axis=1)
print("Predicted class:", pred_labels)
print("Probabilities:", probs)

In [ ]:
predicted_category_name = le.inverse_transform(pred_labels)
print("Predicted category name:", predicted_category_name[0])

In [ ]:
print(preds)

In [ ]:
import os

# Create directory for the classifier
model_path = "./bert_classifier"
if not os.path.exists(model_path):
    os.makedirs(model_path)

# 1. Save the BERT backbone and Tokenizer
# If using DataParallel, we access the underlying model via .module
model_to_save = model.module if hasattr(model, 'module') else model

model_to_save.bert.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

# 2. Save the custom head weights and the label encoder classes
torch.save({
    'fc1_state_dict': model_to_save.fc1.state_dict(),
    'fc2_state_dict': model_to_save.fc2.state_dict(),
    'encoder_classes': le.classes_
}, os.path.join(model_path, "custom_head.pt"))

print(f"Model and tokenizer saved to {model_path}")

In [ ]:
import json

label_map = {int(i): str(cls) for i, cls in enumerate(le.classes_)}
with open(os.path.join(model_path, "label_map.json"), "w", encoding="utf-8") as f:
    json.dump(label_map, f, ensure_ascii=False, indent=2)

print("Saved files:")
for f in os.listdir(model_path):
    print(" -", f)